# Graph of Marks (GoM) - Pipeline Demo

This notebook demonstrates the Graph of Marks pipeline for visual scene understanding.

**Features:**
- Object detection, segmentation, depth estimation, and relationship inference
- Support for custom detection, segmentation, and depth functions
- Flexible hyperparameter tuning
- **GoM Visual Prompting Styles (AAAI 2026 Paper)**

**Contents:**
1. Installation
2. Default Pipeline (YOLOv8 + SAM-HQ + Depth Anything V2)
3. Custom HuggingFace Models (YOLOS + SAM + DPT)
4. Hyperparameter Tuning
5. **GoM Visual Prompting Styles (Paper Configurations)**
6. **Visual + Textual Scene Graph Prompting**

## 1. Installation

Install from TestPyPI (development) or PyPI (stable):

In [ ]:
# Install from TestPyPI (latest development version)
# !pip install --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ "graph-of-marks[all]"

# Install from PyPI (stable release)
# !pip install "graph-of-marks[all]"

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

import gom
print(f"Graph of Marks version: {gom.__version__}")

## Setup

In [ ]:
output_dir = Path("demo_outputs")
output_dir.mkdir(exist_ok=True)

image_path = Path("../images/gqa_sample.jpg")
if not image_path.exists():
    image_path = Path("data/living.png")

if image_path.exists():
    image = Image.open(image_path).convert("RGB")
    W, H = image.size
    print(f"Loaded: {image_path} ({W}x{H})")
    plt.figure(figsize=(10, 8))
    plt.imshow(image)
    plt.axis('off')
    plt.title('Input Image')
    plt.show()
else:
    print(f"Image not found at {image_path}")

## 2. Default Pipeline

The default pipeline uses:
- **Detection:** YOLOv8x
- **Segmentation:** SAM-HQ (vit_h)
- **Depth:** Depth Anything V2 (vitl)

Models are downloaded automatically on first run.

In [ ]:
from gom import Gom

gom_default = Gom(output_dir=str(output_dir / "default"))
result = gom_default.process(image_path)

print(f"Detected {len(result['boxes'])} objects")
print(f"Generated {len(result['masks'])} masks")
print(f"Found {len(result['relationships'])} relationships")
print(f"Processing time: {result['processing_time']:.2f}s")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

outputs = [
    ("01_detections.png", "1. Detections"),
    ("02_segmentation.png", "2. Segmentation"),
    ("03_depth.png", "3. Depth Map"),
    ("04_output.png", "4. Final Output"),
]

for ax, (filename, title) in zip(axes.flat, outputs):
    img_path = output_dir / "default" / f"{image_path.stem}_{filename}"
    if img_path.exists():
        ax.imshow(Image.open(img_path))
    ax.set_title(title, fontsize=14)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 3. Custom HuggingFace Models

Replace any component with your own function. Here we use HuggingFace models:
- **Detection:** YOLOS (hustvl/yolos-tiny)
- **Segmentation:** SAM (facebook/sam-vit-base)
- **Depth:** DPT (Intel/dpt-large)

In [ ]:
import torch
import numpy as np
from typing import List, Tuple

def create_yolos_detector(model_name="hustvl/yolos-tiny", threshold=0.5):
    from transformers import AutoImageProcessor, AutoModelForObjectDetection
    
    device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
    processor = AutoImageProcessor.from_pretrained(model_name)
    model = AutoModelForObjectDetection.from_pretrained(model_name).to(device).eval()
    
    def detect(image: Image.Image) -> Tuple[List, List, List]:
        inputs = processor(images=image, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
        target_sizes = torch.tensor([image.size[::-1]]).to(device)
        results = processor.post_process_object_detection(outputs, target_sizes=target_sizes, threshold=threshold)[0]
        boxes = [box.cpu().tolist() for box in results["boxes"]]
        labels = [model.config.id2label[label.item()] for label in results["labels"]]
        scores = [float(score) for score in results["scores"]]
        return boxes, labels, scores
    return detect


def create_sam_segmenter(model_name="facebook/sam-vit-base"):
    from transformers import SamModel, SamProcessor
    
    device = "cuda" if torch.cuda.is_available() else "cpu"  # SAM has MPS issues
    processor = SamProcessor.from_pretrained(model_name)
    model = SamModel.from_pretrained(model_name).to(device).eval()
    
    def segment(image: Image.Image, boxes: List) -> List[np.ndarray]:
        if not boxes:
            return []
        masks = []
        for box in boxes:
            input_boxes = [[[box[0], box[1], box[2], box[3]]]]
            inputs = processor(image, input_boxes=input_boxes, return_tensors="pt")
            inputs = {k: v.to(device) for k, v in inputs.items()}
            with torch.no_grad():
                outputs = model(**inputs)
            mask = processor.image_processor.post_process_masks(
                outputs.pred_masks.cpu(),
                inputs["original_sizes"].cpu(),
                inputs["reshaped_input_sizes"].cpu()
            )[0][0][0]
            masks.append(mask.numpy().astype(bool))
        return masks
    return segment


def create_dpt_depth(model_name="Intel/dpt-large"):
    from transformers import AutoImageProcessor, AutoModelForDepthEstimation
    
    device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
    processor = AutoImageProcessor.from_pretrained(model_name)
    model = AutoModelForDepthEstimation.from_pretrained(model_name).to(device).eval()
    
    def estimate_depth(image: Image.Image) -> np.ndarray:
        original_size = image.size[::-1]
        inputs = processor(images=image, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
            predicted_depth = outputs.predicted_depth
        prediction = torch.nn.functional.interpolate(
            predicted_depth.unsqueeze(1), size=original_size, mode="bicubic", align_corners=False
        ).squeeze()
        depth = prediction.cpu().numpy()
        depth = (depth - depth.min()) / (depth.max() - depth.min() + 1e-8)
        return 1.0 - depth.astype(np.float32)
    return estimate_depth


print("Creating custom functions...")
yolos_detect = create_yolos_detector(threshold=0.5)
sam_segment = create_sam_segmenter()
dpt_depth = create_dpt_depth()
print("Done!")

In [ ]:
gom_custom = Gom(
    detect_fn=yolos_detect,
    segment_fn=sam_segment,
    depth_fn=dpt_depth,
    output_dir=str(output_dir / "custom"),
)

result_custom = gom_custom.process(image_path)

print(f"Detected {len(result_custom['boxes'])} objects")
print(f"Generated {len(result_custom['masks'])} masks")
print(f"Found {len(result_custom['relationships'])} relationships")
print(f"Processing time: {result_custom['processing_time']:.2f}s")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

titles = [
    (output_dir / "default" / f"{image_path.stem}_01_detections.png", "Default: Detections (YOLOv8)"),
    (output_dir / "custom" / f"{image_path.stem}_01_detections.png", "Custom: Detections (YOLOS)"),
    (output_dir / "default" / f"{image_path.stem}_04_output.png", "Default: Final Output"),
    (output_dir / "custom" / f"{image_path.stem}_04_output.png", "Custom: Final Output"),
]

for ax, (path, title) in zip(axes.flat, titles):
    if path.exists():
        ax.imshow(Image.open(path))
    ax.set_title(title, fontsize=14)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 4. Hyperparameter Tuning

The `Gom` class accepts configuration overrides to control pipeline behavior.

### 4.1 Detection Threshold

In [ ]:
gom_high = Gom(output_dir=str(output_dir / "high_threshold"), threshold_yolo=0.8)
result_high = gom_high.process(image_path)
print(f"High threshold (0.8): {len(result_high['boxes'])} objects")

gom_low = Gom(output_dir=str(output_dir / "low_threshold"), threshold_yolo=0.3)
result_low = gom_low.process(image_path)
print(f"Low threshold (0.3): {len(result_low['boxes'])} objects")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

high_path = output_dir / "high_threshold" / f"{image_path.stem}_01_detections.png"
low_path = output_dir / "low_threshold" / f"{image_path.stem}_01_detections.png"

if high_path.exists():
    axes[0].imshow(Image.open(high_path))
axes[0].set_title(f"High Threshold (0.8): {len(result_high['boxes'])} objects", fontsize=14)
axes[0].axis('off')

if low_path.exists():
    axes[1].imshow(Image.open(low_path))
axes[1].set_title(f"Low Threshold (0.3): {len(result_low['boxes'])} objects", fontsize=14)
axes[1].axis('off')

plt.tight_layout()
plt.show()

### 4.2 Visualization Options

In [ ]:
gom_minimal = Gom(
    output_dir=str(output_dir / "minimal"),
    display_labels=False,
    display_relationships=False,
    show_bboxes=True,
)
result_minimal = gom_minimal.process(image_path)

gom_full = Gom(
    output_dir=str(output_dir / "full"),
    display_labels=True,
    display_relationships=True,
    display_legend=True,
    seg_fill_alpha=0.4,
)
result_full = gom_full.process(image_path)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

minimal_path = output_dir / "minimal" / f"{image_path.stem}_04_output.png"
full_path = output_dir / "full" / f"{image_path.stem}_04_output.png"

if minimal_path.exists():
    axes[0].imshow(Image.open(minimal_path))
axes[0].set_title("Minimal: Boxes only", fontsize=14)
axes[0].axis('off')

if full_path.exists():
    axes[1].imshow(Image.open(full_path))
axes[1].set_title("Full: Labels + Relations + Legend", fontsize=14)
axes[1].axis('off')

plt.tight_layout()
plt.show()

## 5. GoM Visual Prompting Styles (AAAI 2026 Paper)

The Graph-of-Mark paper (AAAI 2026) evaluates different visual prompting configurations.
The library provides **style presets** to easily switch between these configurations:

| Style Preset | Label Mode | Relations | Relation Labels | Best For |
|--------------|------------|-----------|-----------------|----------|
| `som_text` | Textual (oven_1) | ❌ | ❌ | Set-of-Mark baseline |
| `som_numeric` | Numeric (1, 2) | ❌ | ❌ | Set-of-Mark baseline |
| `gom_text` | Textual | ✅ | ❌ | GoM with arrows only |
| `gom_numeric` | Numeric | ✅ | ❌ | GoM with arrows only |
| `gom_text_labeled` | Textual | ✅ | ✅ | **VQA tasks** |
| `gom_numeric_labeled` | Numeric | ✅ | ✅ | **RefCOCO/REC tasks** |

Let's visualize all 6 configurations side-by-side:

In [ ]:
from gom import Gom, GOM_STYLE_PRESETS

# Show available style presets
print("Available GoM Style Presets:")
print("-" * 60)
for style_name, config in GOM_STYLE_PRESETS.items():
    print(f"  {style_name:20s} -> {config}")

In [ ]:
# Process image with all 6 style presets
style_results = {}
styles_to_test = [
    "som_text", "som_numeric",
    "gom_text", "gom_numeric",
    "gom_text_labeled", "gom_numeric_labeled"
]

for style in styles_to_test:
    print(f"Processing with style: {style}...")
    gom_styled = Gom(
        output_dir=str(output_dir / f"style_{style}"),
        style=style
    )
    result = gom_styled.process(image_path)
    style_results[style] = result
    print(f"  ✓ {len(result['boxes'])} objects, {len(result['relationships'])} relationships")

print("\nAll styles processed!")

In [ ]:
# Compare all 6 visual prompting styles
fig, axes = plt.subplots(2, 3, figsize=(20, 14))

style_titles = {
    "som_text": "SoM: Textual IDs\n(oven_1, chair_2)",
    "som_numeric": "SoM: Numeric IDs\n(1, 2, 3)",
    "gom_text": "GoM: Text IDs + Arrows\n(no relation labels)",
    "gom_numeric": "GoM: Numeric IDs + Arrows\n(no relation labels)",
    "gom_text_labeled": "GoM: Text IDs + Labeled Relations\n(BEST for VQA)",
    "gom_numeric_labeled": "GoM: Numeric IDs + Labeled Relations\n(BEST for RefCOCO)",
}

for ax, style in zip(axes.flat, styles_to_test):
    img_path = output_dir / f"style_{style}" / f"{image_path.stem}_04_output.png"
    if img_path.exists():
        ax.imshow(Image.open(img_path))
    ax.set_title(style_titles[style], fontsize=12, fontweight='bold')
    ax.axis('off')

plt.suptitle("GoM Visual Prompting Styles (AAAI 2026 Paper - Table 2)", fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 5.1 Key Differences Between Styles

**Set-of-Mark (SoM) Baselines** - No relationship arrows:
- Objects are labeled but treated as **isolated entities**
- Good baseline for comparing with GoM improvements

**Graph-of-Mark (GoM) with Arrows Only** - Shows connectivity:
- Relationship arrows connect related objects
- Labels are NOT shown on the arrows
- MLM must interpret the arrow direction

**Graph-of-Mark (GoM) with Labeled Relations** - Full information:
- Relationship arrows WITH labels (e.g., "Above", "Left Of")
- **Best overall performance** in the paper's experiments
- Textual IDs work better for VQA, numeric for RefCOCO

## 6. Visual + Textual Scene Graph Prompting

The paper also evaluates **multimodal prompting** where both the visual scene graph (I^SG) 
and a textual description (T^SG) are provided to the MLM.

The library automatically generates textual scene graph representations:

In [ ]:
# Get the result with textual scene graph
gom_multimodal = Gom(
    output_dir=str(output_dir / "multimodal"),
    style="gom_text_labeled",
    include_textual_sg=True  # This is True by default
)
result_mm = gom_multimodal.process(image_path)

print("=" * 70)
print("TEXTUAL SCENE GRAPH REPRESENTATIONS")
print("=" * 70)

# Triples format (for LLM prompts)
print("\n📝 Triples Format (T^SG for LLM prompts):")
print("-" * 50)
print(result_mm["scene_graph_text"])

# Compact prompt format
print("\n📝 Compact Prompt Format:")
print("-" * 50)
print(result_mm["scene_graph_prompt"][:500] + "..." if len(result_mm["scene_graph_prompt"]) > 500 else result_mm["scene_graph_prompt"])

### 6.1 Using Textual SG with Vision-Language Models

Here's how you would use the visual + textual scene graph with a VLM:

In [ ]:
# Example: Building a VQA prompt with Visual + Textual SG
question = "Which object is to the left of the table?"

# Visual SG only prompt (I^SG + T)
visual_only_prompt = f"""Look at the annotated image showing objects and their spatial relationships.

Question: {question}

Answer the question based on the visual annotations in the image."""

# Visual + Textual SG prompt (I^SG + T^SG)
multimodal_prompt = f"""Look at the annotated image showing objects and their spatial relationships.

The scene graph relationships are:
{result_mm["scene_graph_text"]}

Question: {question}

Use both the visual annotations AND the textual relationship descriptions to answer."""

print("=" * 70)
print("VQA PROMPT TEMPLATES")
print("=" * 70)

print("\n🖼️  Visual SG Only (I^SG + T):")
print("-" * 50)
print(visual_only_prompt)

print("\n\n🖼️ + 📝 Visual + Textual SG (I^SG + T^SG):")
print("-" * 50)
print(multimodal_prompt[:800] + "..." if len(multimodal_prompt) > 800 else multimodal_prompt)

### 6.2 Paper Findings on Prompting Modalities

From the paper (Figure 4), the prompting modality comparison shows:

| Prompting Mode | Description | Relative Performance |
|----------------|-------------|---------------------|
| I + T | Raw image + text question | Baseline |
| I + T^SG | Raw image + textual scene graph | +5-8% |
| **I^SG + T** | **Visual scene graph + text question** | **+8-10%** |
| I^SG + T^SG | Visual + textual scene graph | +9-11% |

**Key insight**: Visual scene graph representations (I^SG) consistently outperform 
textual-only encodings, with the combination providing modest additional gains.

## API Summary

### Basic Usage
```python
from gom import Gom

gom = Gom()
result = gom.process("image.jpg")
```

### GoM Style Presets (AAAI 2026 Paper)
```python
from gom import Gom, GOM_STYLE_PRESETS

# Use a style preset matching the paper's experiments
gom = Gom(style="gom_text_labeled")  # Best for VQA
gom = Gom(style="gom_numeric_labeled")  # Best for RefCOCO

# Available styles:
# - "som_text", "som_numeric": Set-of-Mark baselines (no relations)
# - "gom_text", "gom_numeric": GoM with arrows only
# - "gom_text_labeled", "gom_numeric_labeled": GoM with labeled relations
```

### Custom Functions
```python
# detect_fn(image: PIL.Image) -> (boxes, labels, scores)
# segment_fn(image: PIL.Image, boxes: List) -> List[np.ndarray]
# depth_fn(image: PIL.Image) -> np.ndarray

gom = Gom(
    detect_fn=my_detector,
    segment_fn=my_segmenter,
    depth_fn=my_depth,
    style="gom_text_labeled",
)
```

### Key Parameters
| Parameter | Default | Description |
|-----------|---------|-------------|
| `style` | None | GoM style preset (som_text, gom_numeric_labeled, etc.) |
| `label_mode` | "original" | Label format ("original", "numeric", "alphabetic") |
| `display_relationships` | True | Show relationship arrows |
| `display_relation_labels` | True | Show labels on relationship arrows |
| `include_textual_sg` | True | Generate textual scene graph representations |
| `threshold_yolo` | 0.5 | Detection confidence threshold |
| `sam_hq_model_type` | "vit_h" | SAM model size (vit_b, vit_l, vit_h) |
| `display_labels` | True | Show object labels |
| `display_legend` | False | Show color legend |
| `seg_fill_alpha` | 0.25 | Mask transparency (0-1) |

### Output Structure
```python
result = {
    # Detection results
    "boxes": [[x1, y1, x2, y2], ...],
    "labels": ["person", "chair", ...],
    "scores": [0.95, 0.87, ...],
    "masks": [np.ndarray, ...],
    "depth": np.ndarray,
    
    # Relationships
    "relationships": [{"src_idx": 0, "tgt_idx": 1, "relation": "left_of"}, ...],
    
    # Scene graph (for multimodal prompting)
    "scene_graph": nx.DiGraph,  # NetworkX graph object
    "scene_graph_text": "Triples:\nperson ---> (left_of) --> chair\n...",  # T^SG format
    "scene_graph_prompt": "0:person (area=0.15); 1:chair (area=0.08); (0)-left_of->(1)",
    
    # Metadata
    "processing_time": 12.5,
    "output_path": "output/image_04_output.png",
}
```

In [ ]:
print("Demo complete!")
print(f"Outputs saved to: {output_dir.absolute()}")